# 01 — Exploratory Data Analysis & Baseline Error Analysis

**Dataset:** Edge-IIoTset (`ML-EdgeIIoT-dataset.csv`)  
**Goal:** Inspect protocol-level traffic behavior, remove leakage and zero-variance features, establish a leak-free 15-class baseline (XGBoost & LightGBM), and diagnose the primary misclassification bottlenecks.

In [ ]:
!git clone -q https://github.com/PriyoChand/smart-campus-intrusion-detection.git
%cd smart-campus-intrusion-detection

!pip install -q dvc optuna lightgbm xgboost

In [ ]:
import getpass
from google.colab import userdata

DAGSHUB_USER = "PriyoChand"
try:
    DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")
except Exception:
    DAGSHUB_TOKEN = getpass.getpass("DAGsHub Token: ")

!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {DAGSHUB_USER}
!dvc remote modify origin --local password {DAGSHUB_TOKEN}
!dvc pull -q -r origin

## 1. Data Loading & Sanitization

We drop the 17 leakage, session-counter, and raw-payload columns identified in the dataset specification, standardize inactive protocol placeholders (`0.0`, `-`, `?`, etc.) to `"0"` / `0.0`, and remove exact duplicate rows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_PATH = "data/raw/ML-EdgeIIoT-dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape: {df.shape}")

drop_cols = [
    "frame.time", "ip.src_host", "ip.dst_host",
    "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
    "icmp.transmit_timestamp", "udp.stream", "tcp.seq",
    "http.file_data", "http.request.full_uri", "http.request.uri.query",
    "tcp.options", "tcp.payload", "mqtt.msg",
    "tcp.srcport", "tcp.dstport", "udp.port"
]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

target_cols = ["Attack_label", "Attack_type"]
cat_cols = [
    c for c in [
        "http.request.method", "http.referer", "http.request.version",
        "dns.qry.name.len", "mqtt.conack.flags", "mqtt.protoname", "mqtt.topic"
    ] if c in df.columns
]
num_cols = [c for c in df.columns if c not in cat_cols + target_cols]

# Normalize inactive protocol tokens while preserving valid strings (e.g., "MQTT")
inactive_tokens = ["0.0", "-", ";", "?", "None", "nan", ""]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().replace(inactive_tokens, "0")

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

n_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dropped {n_before - len(df):,} duplicate rows -> Clean shape: {df.shape}")
print(f"Remaining NaNs: {df.isna().sum().sum()}")

## 2. Exploratory Data Analysis

### 2.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(
    data=df, x="Attack_label", hue="Attack_label",
    ax=axes[0], palette=["#2E8540", "#C53030"], legend=False
)
axes[0].set_title("Binary Distribution: Normal vs Attack", fontweight="bold")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Normal (0)", "Attack (1)"])
axes[0].set_ylabel("Flows")

order = df["Attack_type"].value_counts().index
sns.countplot(
    data=df, y="Attack_type", hue="Attack_type",
    order=order, ax=axes[1], palette="viridis", legend=False
)
axes[1].set_title("Multi-Class Distribution (15 Traffic Types)", fontweight="bold")
axes[1].set_xlabel("Flows")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

class_summary = pd.DataFrame({
    "Count": df["Attack_type"].value_counts(),
    "Share (%)": (df["Attack_type"].value_counts(normalize=True) * 100).round(2)
})
display(class_summary)

### 2.2 Zero-Variance Audit & Numerical Correlation

In [ ]:
zero_var_cols = [c for c in num_cols if df[c].nunique() <= 1]
print(f"Dropping {len(zero_var_cols)} zero-variance numerical columns: {zero_var_cols}")

if zero_var_cols:
    df.drop(columns=zero_var_cols, inplace=True)
    num_cols = [c for c in num_cols if c not in zero_var_cols]

corr_target = df[num_cols + ["Attack_label"]].corr()["Attack_label"].abs().sort_values(ascending=False)
top_num = corr_target.head(13).index.tolist()

plt.figure(figsize=(12, 8))
corr_matrix = df[top_num].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5
)
plt.title("Top 12 Numerical Features Correlated with Attack_label", fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 Protocol-Level Behavior (MQTT, DNS, Active HTTP)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

top_mqtt = df["mqtt.hdrflags"].astype(str).value_counts().nlargest(5).index
sns.countplot(
    data=df[df["mqtt.hdrflags"].astype(str).isin(top_mqtt)],
    x="mqtt.hdrflags", hue="Attack_label",
    palette={0: "#2E8540", 1: "#C53030"}, ax=axes[0]
)
axes[0].set_yscale("log")
axes[0].set_title("MQTT Header Flags (Log Scale)", fontweight="bold")
axes[0].set_xlabel("mqtt.hdrflags (0.0 = Non-MQTT)")
axes[0].set_ylabel("Packet Count")

top_dns = df["dns.qry.name.len"].value_counts().nlargest(5).index
sns.countplot(
    data=df[df["dns.qry.name.len"].isin(top_dns)],
    x="dns.qry.name.len", hue="Attack_label",
    palette={0: "#2E8540", 1: "#C53030"}, ax=axes[1]
)
axes[1].set_yscale("log")
axes[1].set_title("Top 5 DNS Query Values (Log Scale)", fontweight="bold")
axes[1].set_xlabel("dns.qry.name.len ('0' = Non-DNS)")
axes[1].set_ylabel("Packet Count")
axes[1].tick_params(axis="x", rotation=20)

active_http = df[df["http.content_length"] > 0]
sns.boxplot(
    data=active_http, x="http.content_length", y="Attack_type",
    hue="Attack_type", palette="magma", ax=axes[2], legend=False
)
axes[2].set_xscale("log")
axes[2].set_title("Active HTTP Payload Sizes by Attack Type", fontweight="bold")
axes[2].set_xlabel("http.content_length (Bytes, Log Scale)")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

## 3. Preprocessing & Stratified Train/Test Split

Because rare classes like `MITM` (0.08%) and `Fingerprinting` (0.61%) are heavily outnumbered, we use a stratified 80/20 split and evaluate primarily on **Macro F1**. Imputers and unknown-category handling are kept inside `ColumnTransformer` so the pipeline stays safe for unseen inference traffic.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

zero_var_cats = [c for c in cat_cols if df[c].nunique() <= 1]
if zero_var_cats:
    df.drop(columns=zero_var_cats, inplace=True)
    cat_cols = [c for c in cat_cols if c not in zero_var_cats]

X = df.drop(columns=target_cols)
le = LabelEncoder()
y = le.fit_transform(df["Attack_type"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="0")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)
preprocessor.set_output(transform="pandas")

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Active features: {len(num_cols)} numerical, {len(cat_cols)} categorical ({X_train_proc.shape[1]} total)")
print(f"Train: {X_train_proc.shape} | Test: {X_test_proc.shape}")

## 4. Baseline Modeling (XGBoost, LightGBM & Optuna)

We compare default XGBoost, default LightGBM, and Optuna-tuned XGBoost on the 36 clean features to test whether performance is model-limited or feature-limited.

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

results = []

def evaluate_model(name, y_true, y_pred, show_report=False):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Macro F1": round(macro_f1, 4),
        "Weighted F1": round(weighted_f1, 4)
    })
    print(f"[{name}] Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f}")
    if show_report:
        print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

xgb_base = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1, eval_metric="mlogloss"
)
xgb_base.fit(X_train_proc, y_train)
y_pred_xgb = xgb_base.predict(X_test_proc)
evaluate_model("XGBoost (Baseline)", y_test, y_pred_xgb, show_report=True)

lgb_base = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_base.fit(X_train_proc, y_train)
y_pred_lgb = lgb_base.predict(X_test_proc)
evaluate_model("LightGBM (Baseline)", y_test, y_pred_lgb)

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss"
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in cv.split(X_train_proc, y_train):
        model = XGBClassifier(**params)
        model.fit(X_train_proc.iloc[tr_idx], y_train[tr_idx])
        preds = model.predict(X_train_proc.iloc[val_idx])
        scores.append(f1_score(y_train[val_idx], preds, average="macro"))
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, timeout=600)

xgb_opt = XGBClassifier(**study.best_params, random_state=42, n_jobs=-1, eval_metric="mlogloss")
xgb_opt.fit(X_train_proc, y_train)
y_pred_opt = xgb_opt.predict(X_test_proc)
evaluate_model("XGBoost (Optuna)", y_test, y_pred_opt)

display(pd.DataFrame(results))

## 5. Error Analysis & Feature Importance

In [ ]:
# Top 15 feature importances (Baseline XGBoost)
feat_imp = pd.DataFrame({
    "Feature": X_train_proc.columns,
    "Importance (%)": (xgb_base.feature_importances_ * 100).round(2)
}).sort_values("Importance (%)", ascending=False)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=feat_imp.head(15), x="Importance (%)", y="Feature",
    hue="Feature", palette="mako", legend=False
)
plt.title("Top 15 Feature Importances (Baseline XGBoost — 36 Features)", fontweight="bold")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Derived binary detection (Normal vs Any Attack)
normal_id = le.transform(["Normal"])[0]
y_test_bin = (y_test != normal_id).astype(int)
y_pred_bin = (y_pred_xgb != normal_id).astype(int)
print(
    f"Derived Binary Detection (Normal vs Attack) -> "
    f"Accuracy: {accuracy_score(y_test_bin, y_pred_bin)*100:.2f}%, "
    f"F1: {f1_score(y_test_bin, y_pred_bin):.4f}"
)

# 15-class confusion matrix
plt.figure(figsize=(14, 10))
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_
)
plt.title("15-Class Confusion Matrix (Baseline XGBoost)", fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Key Findings & Next Steps

* **Model-Agnostic ~86% Ceiling:** Default XGBoost (86.05% Acc / 0.8753 Macro F1), LightGBM (85.91% / 0.8727), and Optuna-tuned XGBoost (86.00% / 0.8749) all converge at the same ceiling, indicating a feature information bottleneck rather than underfitting.
* **Primary Error Concentration:** ~80% of all misclassifications land in `Normal` and `DDoS_HTTP`. Web attacks (`SQL_injection`, `XSS`, `Password`, `Uploading`) and reconnaissance traffic (`Fingerprinting`) lose their distinguishing signatures once raw HTTP/TCP payload strings and port columns are dropped.
* **Multicollinearity:** `mqtt.hdrflags` / `mqtt.msgtype` and `mqtt.ver` / `mqtt.conflags` / `mqtt.conflag.cleansess` / `mqtt.proto_len` are 100% collinear (`r = 1.00`), splitting tree importance when `colsample_bytree < 1`.
* **Next (`02_Feature_Engineering_and_XAI.ipynb`):** Extract non-leaky structural metadata (URI/payload character lengths and destination port tiers) *before* dropping raw strings/ports, drop `r = 1.00` clones, re-benchmark XGBoost/LightGBM, and run SHAP explainability.